In [331]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

SHOW_GRAPHS = True

# Preparation of Data Frames

In [332]:
df  = pd.DataFrame()

## Sentiment

In [333]:
# Reading the saved data pickle file

df_sentiment  = pd.DataFrame()

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl', compression=None)
# df_sentiment.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_sentiment.drop(columns=["Company"], inplace=True)

df_sentiment.index = pd.to_datetime(df_sentiment.index)
df_sentiment.Date = pd.to_datetime(df_sentiment.Date)

df_sentiment = df_sentiment.ffill()

df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment"].shift(periods=1)
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment"].shift(periods=2)
df_sentiment["sentiment(t-1)"] = df_sentiment["sentiment(t-1)"].ffill()
df_sentiment["sentiment(t-2)"] = df_sentiment["sentiment(t-2)"].ffill()

# SENTIMENT RETURN
df_sentiment["sentiment_return"]=(df_sentiment["sentiment(t-1)"]-df_sentiment["sentiment(t-2)"])/df_sentiment["sentiment(t-2)"]
# SENTIMENT LN RETURN
df_sentiment["sentiment_ln_return"]=np.log(df_sentiment["sentiment(t-1)"]/df_sentiment["sentiment(t-2)"])
# SENTIMENT EWM
df_sentiment["sentiment_ewm"]=df_sentiment["sentiment"].ewm(10).mean()

In [334]:
df_sentiment = df_sentiment[["Date", "sentiment", "sentiment(t-1)", "sentiment(t-2)", "sentiment_return", "sentiment_ln_return", "sentiment_ewm", "PressReleaseCount"]]

In [335]:
df = df_sentiment

In [336]:
df

,Date,sentiment,sentiment(t-1),sentiment(t-2),sentiment_return,sentiment_ln_return,sentiment_ewm,PressReleaseCount
2019-09-25,2019-09-25,94.117647,NaN,NaN,NaN,NaN,94.117647,13
2019-09-26,2019-09-26,85.826772,94.117647,NaN,NaN,NaN,89.774808,11
2019-09-27,2019-09-27,94.545455,85.826772,94.117647,-0.088091,-0.092215,91.518760,7
2019-09-30,2019-09-30,90.000000,94.545455,85.826772,0.101585,0.096750,91.083192,10
2019-10-01,2019-10-01,100.000000,90.000000,94.545455,-0.048077,-0.049271,93.221585,12
...,...,...,...,...,...,...,...,...
2021-08-19,2021-08-19,71.779141,68.047337,81.595092,-0.166036,-0.181566,77.598536,25
2021-08-20,2021-08-20,74.545455,71.779141,68.047337,0.054841,0.053390,77.320983,13
2021-08-23,2021-08-23,74.698795,74.545455,71.779141,0.038539,0.037815,77.082602,29
2021-08-24,2021-08-24,85.326087,74.698795,74.545455,0.002057,0.002055,77.832010,30


## VIX

In [337]:
from yahoo_fin.stock_info import get_data

df_vix  = pd.DataFrame()
df_vix = get_data("^VIX", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_vix.index = pd.to_datetime(df_vix.index)
df_vix["Date"]= pd.to_datetime(df_vix.index)

df_vix["vix"] = df_vix["adjclose"]
df_vix.drop(columns=["high", "low", "adjclose", "ticker", "volume"], inplace=True)
df_vix["vix"]  = df_vix["vix"].ffill()

df_vix["vix_yesterday"] = df_vix["vix"].shift(periods=1)
# vix RETURN
df_vix["vix_return_c2c"]=(df_vix["vix"]-df_vix["vix_yesterday"])/df_vix["vix_yesterday"]
df_vix["vix_return_c2c"]=(df_vix["close"]-df_vix["open"])/df_vix["open"]
# vix LN RETURN
df_vix["vix_ln_return_c2c"]=np.log(df_vix["vix"]/df_vix["vix_yesterday"])
df_vix["vix_ln_return"]=np.log(df_vix["close"]/df_vix["open"])

# vix EWM
df_vix["vix_ewm"]=df_vix["vix"].ewm(10).mean()

In [338]:
df_vix

,open,close,Date,vix,vix_yesterday,vix_return_c2c,vix_ln_return_c2c,vix_ln_return,vix_ewm
2019-09-25,17.049999,15.960000,2019-09-25,15.960000,NaN,-0.063930,NaN,-0.066065,15.960000
2019-09-26,16.230000,16.070000,2019-09-26,16.070000,15.960000,-0.009858,0.006869,-0.009907,16.017619
2019-09-27,15.770000,17.219999,2019-09-27,17.219999,16.070000,0.091947,0.069117,0.087962,16.457160
2019-09-30,17.230000,16.240000,2019-09-30,16.240000,17.219999,-0.057458,-0.058594,-0.059175,16.394880
2019-10-01,16.020000,18.559999,2019-10-01,18.559999,16.240000,0.158552,0.133531,0.147171,16.914110
...,...,...,...,...,...,...,...,...,...
2021-08-19,23.120001,21.670000,2021-08-19,21.670000,21.570000,-0.062716,0.004625,-0.064769,17.871938
2021-08-20,22.740000,18.559999,2021-08-20,18.559999,21.670000,-0.183817,-0.154920,-0.203117,17.934489
2021-08-23,18.830000,17.150000,2021-08-23,17.150000,18.559999,-0.089219,-0.079011,-0.093453,17.863172
2021-08-24,16.959999,17.219999,2021-08-24,17.219999,17.150000,0.015330,0.004073,0.015214,17.804702


In [339]:
df = pd.merge(df_vix,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

## S&P500

In [340]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")

In [341]:
df_sp500 = get_data("^GSPC", start_date="25/09/2019", end_date="26/08/2021", index_as_date = True, interval="1d")
df_sp500.index = pd.to_datetime(df_sp500.index)
df_sp500["Date"]= pd.to_datetime(df_sp500.index)

df_sp500["sp500"] = df_sp500["adjclose"]
df_sp500.drop(columns=["open", "high", "low", "close", "adjclose", "ticker", "volume"], inplace=True)
df_sp500["sp500"]  = df_sp500["sp500"].ffill()

df_sp500["sp500_yesterday"] = df_sp500["sp500"].shift(periods=1)
# sp500 RETURN
df_sp500["sp500_return"]=(df_sp500["sp500"]-df_sp500["sp500_yesterday"])/df_sp500["sp500_yesterday"]
# sp500 LN RETURN
df_sp500["sp500_ln_return"]=np.log(df_sp500["sp500"]/df_sp500["sp500_yesterday"])
# sp500 EWM
df_sp500["sp500_ewm"]=df_sp500["sp500"].ewm(10).mean()

In [342]:
df = pd.merge(df_sp500,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [343]:
df

,Date,sp500,sp500_yesterday,sp500_return,sp500_ln_return,sp500_ewm,open,close,vix,vix_yesterday,...,vix_ln_return_c2c,vix_ln_return,vix_ewm,sentiment,sentiment(t-1),sentiment(t-2),sentiment_return,sentiment_ln_return,sentiment_ewm,PressReleaseCount
0,2019-09-25,2984.870117,NaN,NaN,NaN,2984.870117,17.049999,15.960000,15.960000,NaN,...,NaN,-0.066065,15.960000,94.117647,NaN,NaN,NaN,NaN,94.117647,13.0
1,2019-09-26,2977.620117,2984.870117,-0.002429,-0.002432,2981.072498,16.230000,16.070000,16.070000,15.960000,...,0.006869,-0.009907,16.017619,85.826772,94.117647,NaN,NaN,NaN,89.774808,11.0
2,2019-09-27,2961.790039,2977.620117,-0.005316,-0.005331,2974.023623,15.770000,17.219999,17.219999,16.070000,...,0.069117,0.087962,16.457160,94.545455,85.826772,94.117647,-0.088091,-0.092215,91.518760,7.0
3,2019-09-30,2976.739990,2961.790039,0.005048,0.005035,2974.802655,17.230000,16.240000,16.240000,17.219999,...,-0.058594,-0.059175,16.394880,90.000000,94.545455,85.826772,0.101585,0.096750,91.083192,10.0
4,2019-10-01,2940.250000,2976.739990,-0.012258,-0.012334,2966.516379,16.020000,18.559999,18.559999,16.240000,...,0.133531,0.147171,16.914110,100.000000,90.000000,94.545455,-0.048077,-0.049271,93.221585,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,2021-08-19,4405.799805,4400.270020,0.001257,0.001256,4413.914368,23.120001,21.670000,21.670000,21.570000,...,0.004625,-0.064769,17.871938,71.779141,68.047337,81.595092,-0.166036,-0.181566,77.598536,25.0
480,2021-08-20,4441.669922,4405.799805,0.008142,0.008109,4416.437600,22.740000,18.559999,18.559999,21.670000,...,-0.154920,-0.203117,17.934489,74.545455,71.779141,68.047337,0.054841,0.053390,77.320983,13.0
481,2021-08-23,4479.529785,4441.669922,0.008524,0.008488,4422.173254,18.830000,17.150000,17.150000,18.559999,...,-0.079011,-0.093453,17.863172,74.698795,74.545455,71.779141,0.038539,0.037815,77.082602,29.0
482,2021-08-24,4486.229980,4479.529785,0.001496,0.001495,4427.996592,16.959999,17.219999,17.219999,17.150000,...,0.004073,0.015214,17.804702,85.326087,74.698795,74.545455,0.002057,0.002055,77.832010,30.0


In [344]:
# df = df.drop([475])

# DAY MONTH YEAR

In [345]:
import datetime as dt
from datetime import date
import calendar
my_date = date.today()
calendar.day_name[my_date.weekday()]  #'Wednesday'

# Prepare data
df['year'] = [d.year for d in df.Date]
df['month'] = [d.strftime('%b') for d in df.Date]
df['day'] = [d.day_name() for d in df.Date]
years = df['year'].unique()
df['month-year'] = [d.strftime('%b') + str(d.year) for d in df.Date]

# TEXTBLOB

In [346]:
# Reading the saved data pickle file

df_polarity = pd.DataFrame()

df_polarity=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentArticlesTextBlob.pkl', compression=None)
# df_polarity.drop(columns=["compound", "neg", "pos", "neu", "Articles", "Title", "Text", "Company"], inplace=True)
df_polarity.drop(columns=["Company", "PressReleaseCount", "Articles", "Title", "Text"], inplace=True)

df_polarity.index = pd.to_datetime(df_polarity.index)
df_polarity.Date = pd.to_datetime(df_polarity.Date)

df_polarity = df_polarity.ffill()

df_polarity["subjectivity(t-1)"] = df_polarity["subjectivity"].shift(periods=1)
df_polarity["subjectivity(t-2)"] = df_polarity["subjectivity"].shift(periods=2)

# polarity RETURN
df_polarity["subjectivity_return"]=(df_polarity["subjectivity(t-1)"]-df_polarity["subjectivity(t-2)"])/df_polarity["subjectivity(t-2)"]
# polarity LN RETURN
df_polarity["subjectivity_ln_return"]=np.log(df_polarity["subjectivity(t-1)"]/df_polarity["subjectivity(t-2)"])
# polarity EWM
df_polarity["subjectivity_ewm"]=df_polarity["subjectivity"].ewm(10).mean()

In [347]:
df.reset_index(drop=True, inplace=True)
df_polarity.reset_index(drop=True, inplace=True)
df_polarity = df_polarity.ffill()

In [348]:
df = pd.merge(df_polarity,df,how='outer', on='Date')
df = df.ffill()
df = df.sort_values(by="Date")

In [349]:
df.index = df.Date

In [350]:
df

,polarity,subjectivity,PosNegNeu,Date,subjectivity(t-1),subjectivity(t-2),subjectivity_return,subjectivity_ln_return,subjectivity_ewm,sp500,...,sentiment(t-1),sentiment(t-2),sentiment_return,sentiment_ln_return,sentiment_ewm,PressReleaseCount,year,month,day,month-year
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,0.022975,0.254230,Positive,2019-09-25,NaN,NaN,NaN,NaN,0.254230,2984.870117,...,NaN,NaN,NaN,NaN,94.117647,13.0,2019.0,Sep,Wednesday,Sep2019
2019-09-26,0.017622,0.266415,Positive,2019-09-26,0.254230,NaN,NaN,NaN,0.260612,2977.620117,...,94.117647,NaN,NaN,NaN,89.774808,11.0,2019.0,Sep,Thursday,Sep2019
2019-09-27,0.047938,0.412416,Positive,2019-09-27,0.266415,0.254230,0.047930,0.046816,0.316106,2961.790039,...,85.826772,94.117647,-0.088091,-0.092215,91.518760,7.0,2019.0,Sep,Friday,Sep2019
2019-09-30,0.031246,0.367310,Positive,2019-09-30,0.412416,0.266415,0.548021,0.436977,0.330791,2976.739990,...,94.545455,85.826772,0.101585,0.096750,91.083192,10.0,2019.0,Sep,Monday,Sep2019
2019-10-01,0.105016,0.415282,Positive,2019-10-01,0.367310,0.412416,-0.109370,-0.115826,0.351053,2940.250000,...,90.000000,94.545455,-0.048077,-0.049271,93.221585,12.0,2019.0,Oct,Tuesday,Oct2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-25,0.137566,0.335002,Positive,2021-08-25,0.339332,0.339064,0.000793,0.000792,0.340351,4496.189941,...,85.326087,74.698795,0.142269,0.133016,78.518610,26.0,2021.0,Aug,Wednesday,Aug2021
2021-08-26,0.060228,0.344194,Positive,2021-08-26,0.335002,0.339332,-0.012762,-0.012844,0.340701,4496.189941,...,85.326087,74.698795,0.142269,0.133016,78.518610,26.0,2021.0,Aug,Wednesday,Aug2021
2021-08-27,0.102358,0.352568,Positive,2021-08-27,0.344194,0.335002,0.027440,0.027070,0.341780,4496.189941,...,85.326087,74.698795,0.142269,0.133016,78.518610,26.0,2021.0,Aug,Wednesday,Aug2021


# THREE PERIODS

In [351]:
df = df.ffill()
df.index = df.Date

In [352]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-08-31']
df_part3 = df_part3.ffill()

In [ ]:
df_part3

In [353]:
# df_part3["sentiment(t-1)"] = df_part3["sentiment"].shift(periods=1)
# df_part3["sentiment(t-2)"] = df_part3["sentiment"].shift(periods=2)


# # SENTIMENT RETURN
# df_part3["sentiment_return"]=(df_part3["sentiment(t-1)"]-df_part3["sentiment(t-2)"])/df_part3["sentiment(t-2)"]
# # SENTIMENT LN RETURN
# df_part3["sentiment_ln_return"]=np.log(df_part3["sentiment(t-1)"]/df_part3["sentiment(t-2)"])
# # SENTIMENT EWM
# df_part3["sentiment_ewm"]=df_part3["sentiment"].ewm(10).mean()

# DF HOURLY

In [354]:
df_time=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesTime.pkl', compression=None)
df_hour=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticlesDateHour.pkl', compression=None)

In [355]:
df_hour.Date= pd.to_datetime(df_hour.Date)

In [356]:
df_hour = df_hour.merge(df_sp500, on='Date', how='left')

In [357]:
# Prepare data
df_hour['year'] = [d.year for d in df_hour.Date]
df_hour['month'] = [d.strftime('%b') for d in df_hour.Date]
df_hour['day'] = [d.day_name() for d in df_hour.Date]
years = df_hour['year'].unique()
df_hour['month-year'] = [d.strftime('%b') + str(d.year) for d in df_hour.Date]

In [358]:
df_hour.index = df_hour.Date

In [359]:
df_hour.sort_values(by="Hour")

,Date,Hour,compound,neg,neu,pos,sentiment,Articles,Title,Text,...,PressReleaseCount,sp500,sp500_yesterday,sp500_return,sp500_ln_return,sp500_ewm,year,month,day,month-year
Date,,,,,,,,,,,,,,,,,,,,,
2019-10-15,2019-10-15,0.0,0.8860,0.045,0.768,0.187,80.603448,. WEWORK PREFERS JPMORGAN FINANCING LIFELINE T...,. WEWORK PREFERS JPMORGAN FINANCING LIFELINE T...,. :WEWORK PREFERS JPMORGAN FINANCING LIFELINE ...,...,2,2995.679932,2966.149902,0.009956,0.009906,2948.610047,2019,Oct,Tuesday,Oct2019
2021-04-14,2021-04-14,0.0,0.0000,0.000,1.000,0.000,0.000000,. NISOURCE INC. ANNOUNCES PRICING OF EQUITY UN...,. NISOURCE INC. ANNOUNCES PRICING OF EQUITY UNITS,". MERRILLVILLE, IND., APRIL 14, 2021 /PRNEWSWI...",...,1,4124.660156,4141.589844,-0.004088,-0.004096,4028.101781,2021,Apr,Wednesday,Apr2021
2021-04-12,2021-04-12,0.0,0.2732,0.078,0.828,0.094,54.651163,. PHASE 3 PREVENTION TRIAL SHOWED 81% REDUCED ...,. PHASE 3 PREVENTION TRIAL SHOWED 81% REDUCED ...,". TARRYTOWN, N.Y., APRIL 12, 2021 /PRNEWSWIRE/...",...,1,4127.990234,4128.799805,-0.000196,-0.000196,4006.131553,2021,Apr,Monday,Apr2021
2021-04-09,2021-04-09,0.0,0.0000,0.000,1.000,0.000,0.000000,. NYXOAH REPORTS FULL YEAR 2020 RESULTS. NYXOA...,. NYXOAH REPORTS FULL YEAR 2020 RESULTS,. NYXOAH REPORTS FULL YEAR 2020 RESULTS,...,1,4128.799805,4097.169922,0.007720,0.007690,3993.945685,2021,Apr,Friday,Apr2021
2021-04-07,2021-04-07,0.0,0.0258,0.074,0.860,0.066,47.142857,. INDUSTRY'S FIRST DC/DC CONTROLLERS WITH AN I...,. INDUSTRY'S FIRST DC/DC CONTROLLERS WITH AN I...,". DALLAS, APRIL 7, 2021 /PRNEWSWIRE/ -- TEXAS ...",...,1,4079.949951,4073.939941,0.001475,0.001474,3968.789308,2021,Apr,Wednesday,Apr2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-05,2021-08-05,NaN,0.0000,0.000,0.000,0.000,0.000000,,,,...,0,4429.100098,4402.660156,0.006005,0.005987,4375.213986,2021,Aug,Thursday,Aug2021
2021-08-06,2021-08-06,NaN,0.0000,0.000,0.000,0.000,0.000000,,,,...,0,4436.520020,4429.100098,0.001675,0.001674,4380.787262,2021,Aug,Friday,Aug2021
2021-08-10,2021-08-10,NaN,0.0000,0.000,0.000,0.000,0.000000,,,,...,0,4436.750000,4432.350098,0.000993,0.000992,4390.136175,2021,Aug,Tuesday,Aug2021


In [360]:
df_hour_part1 = df_hour.loc['2019-09-25':'2020-02-14']
df_hour_part2 = df_hour.loc['2020-02-15':'2020-05-31']
df_hour_part3 = df_hour.loc['2020-06-01':'2021-08-31']

In [361]:
df_hour["sentiment"]

Date
2019-09-25      0.000000
2019-09-26      0.000000
2019-09-27      0.000000
2019-09-30      0.000000
2019-10-01      0.000000
                 ...    
2021-08-09     84.466019
2021-08-10      0.000000
2021-08-11    100.000000
2021-08-12      0.000000
2021-08-13      0.000000
Name: sentiment, Length: 10920, dtype: float64

# 09-16

In [362]:
df_09_15=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_09_15.pkl', compression=None)
df_16_08=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles_16_08.pkl', compression=None)
df_09_15.index= pd.to_datetime(df_09_15.index)
df_16_08.index= pd.to_datetime(df_16_08.index)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [363]:
# Prepare data df_09_15
df_09_15['year'] = [d.year for d in df_09_15.Date]
df_09_15['month'] = [d.strftime('%b') for d in df_09_15.Date]
df_09_15['day'] = [d.day_name() for d in df_09_15.Date]
years = df_09_15['year'].unique()
df_09_15['month-year'] = [d.strftime('%b') + str(d.year) for d in df_09_15.Date]

# Prepare data df_16_08
df_16_08['year'] = [d.year for d in df_16_08.Date]
df_16_08['month'] = [d.strftime('%b') for d in df_16_08.Date]
df_16_08['day'] = [d.day_name() for d in df_16_08.Date]
years = df_16_08['year'].unique()
df_16_08['month-year'] = [d.strftime('%b') + str(d.year) for d in df_16_08.Date]


In [364]:
df_09_15 = df_09_15.merge(df_vix,on='Date',how='left')
df_09_15 = df_09_15.merge(df_sp500, on='Date', how='left')

df_16_08 = df_16_08.merge(df_vix,on='Date',how='left')
df_16_08 = df_16_08.merge(df_sp500, on='Date', how='left')

In [365]:
df_09_15.index= pd.to_datetime(df_09_15.Date)
df_16_08.index= pd.to_datetime(df_16_08.Date)
df_09_15.Date= pd.to_datetime(df_09_15.Date)
df_16_08.Date= pd.to_datetime(df_16_08.Date)

In [366]:
df_09_15["sentiment"] = df_09_15["sentiment"].apply(np.float)
df_16_08["sentiment"] = df_16_08["sentiment"].apply(np.float)

In [367]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_09_15["sentiment_ln_return"]=np.log(df_09_15["sentiment"]/sentiment_yesterday)


sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
sentiment_yesterday = sentiment_yesterday.ffill()
df_16_08["sentiment_ln_return"]=np.log(df_16_08["sentiment"]/sentiment_yesterday)

In [368]:
df_09_15["sentiment"] = df_09_15["sentiment"].ffill()
df_16_08["sentiment"] = df_16_08["sentiment"].ffill()

In [369]:
sentiment_yesterday = df_09_15["sentiment"].shift(periods=1)
df_09_15["sentiment_return"]=((df_09_15["sentiment"]-sentiment_yesterday)/sentiment_yesterday)
sentiment_yesterday = df_16_08["sentiment"].shift(periods=1)
df_16_08["sentiment_return"]=((df_16_08["sentiment"]-sentiment_yesterday)/sentiment_yesterday)

In [370]:
df_09_15["sentiment_return"] = df_09_15["sentiment_return"].ffill()
df_16_08["sentiment_return"] = df_16_08["sentiment_return"].ffill()

In [371]:
# df_part3['Num_word'] = df_part3['Articles'].apply(lambda x:len(str(x).split())) #Number Of words in main text